In [224]:
import torch
from PAF_model import *
import pandas as pd
import numpy as np
#import optuna
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model takes as input an ESM embedding matrix of N x M (where N is number of proteins and M is fixed size embedding dimension)
# Edges will be directly fed in shape of [2, N] where N is number of edges
# Edge weight tensor of [N] where N is number of edges


# Synthetic data generation below

In [ ]:
# Sample dataset: 200 random nodes with fixed embedding size 128
n = 200
esm_dim = 128
input_data = torch.randn((n, esm_dim))


In [ ]:
# Generate random dataset of (weighted) undirected edges of nodes
# Edges start in shape [# of edges, 2] but are fed into model as [2, # of edges]
def generate_power_law_indices(n, num_unique_indices, alpha=2.0):
    """
    n: total number of pairs to generate
    num_unique_indices: the range of indices (0 to max_idx)
    alpha: the power law exponent (higher = more skewed)
    """
    # 1. Generate probabilities following a Power Law (1 / x^alpha)
    weights = 1.0 / (torch.arange(1, num_unique_indices+1).float() ** alpha)
    
    # 2. Multinomial sampling: Pick '2*n' indices based on these weights
    # replacement=True is necessary to allow 'hubs' to appear many times
    sampled_indices = torch.multinomial(weights, num_samples=2*n, replacement=True)
    
    # 3. Reshape into [n, 2]
    tensor_2d = sampled_indices.view(n, 2)
    
    return tensor_2d

def generate_undirected_unique_power_law(n_target, num_nodes, alpha=1.5):
    # 1. Generate more pairs than needed (since we'll prune repeats)
    # We generate ~2x to ensure we have enough after pruning
    raw_indices = generate_power_law_indices(n_target * 100, num_nodes, alpha)
    
    # 2. Sort each pair (row-wise) so [5, 1] becomes [1, 5]
    # This treats undirected edges identically
    sorted_edges, _ = torch.sort(raw_indices, dim=1)
    
    # 3. Remove self-loops (where index1 == index2)
    mask = sorted_edges[:, 0] != sorted_edges[:, 1]
    sorted_edges = sorted_edges[mask]
    
    # 4. Find unique rows
    unique_edges = torch.unique(sorted_edges, dim=0)
    
    # 5. Trim to your target 'n' (if you generated too many)
    if unique_edges.size(0) > n_target:
        unique_edges = unique_edges[:n_target]
        
    return unique_edges

input_edges = generate_undirected_unique_power_law(100000, n, 2.5)
edge_weights = torch.abs(torch.randn(input_edges.shape[0]))
edge_weights = (edge_weights - torch.min(edge_weights))/(torch.max(edge_weights)-torch.min(edge_weights))


In [129]:
print(input_edges.shape)
print(edge_weights.shape)

torch.Size([2082, 2])
torch.Size([2082])


In [ ]:
# Count to identify nodes with the  most edges

index_count = torch.bincount(input_edges.flatten(), minlength = n)

# Arbitrary split of train/valid/test sets

In [ ]:
# Arbitrary division of train/valid/test split
# Keep most connected nodes since these are a prior known already
# Train set - keep edges with only nodes in training set
# Test set - keep edges where at least one node is in the test set
# Validation set - all other edges


keep_top_n = 10
train_prop = 0.8
test_prop = 0.075
val_prop = 1 - train_prop - test_prop

split1 = int(np.floor(n*train_prop - keep_top_n))
split2 = int(np.floor(n*(1-test_prop) - keep_top_n))


In [178]:
idx = torch.randperm(n - keep_top_n)

######
remaining_idx = torch.arange(keep_top_n,n)
train_idx = remaining_idx[idx[:split1]]
val_idx = remaining_idx[idx[split1:split2]]
test_idx = remaining_idx[idx[split2:]]

train_idx = torch.cat((torch.arange(10), train_idx), dim = -1)
train_idx = torch.sort(train_idx)[0]
val_idx = torch.sort(val_idx)[0]
test_idx = torch.sort(test_idx)[0]

##########

mask_col0 = torch.isin(input_edges[:, 0], train_idx)
mask_col1 = torch.isin(input_edges[:, 1], train_idx)

train_mask = mask_col0 & mask_col1
train_edges = input_edges[train_mask].t()
train_weights = edge_weights[train_mask]
print(train_edges.shape)

mask_col0 = torch.isin(input_edges[:, 0], test_idx)
mask_col1 = torch.isin(input_edges[:, 1], test_idx)

test_mask = mask_col0 | mask_col1
test_edges = input_edges[test_mask].t()
test_weights = edge_weights[test_mask]
print(test_edges.shape)

val_mask = ~(test_mask | train_mask)
val_edges = input_edges[val_mask].t()
val_weights = edge_weights[val_mask]
print(val_edges.shape)

########
### Here potentially allow shuffles of train/validation for
train_val_edges = torch.cat((train_edges,val_edges), dim = -1)
train_val_edge_weights = torch.cat((train_weights, val_weights), dim = -1)

all_edges = torch.cat((train_edges,val_edges, test_edges), dim = -1)
all_edge_weights = torch.cat((train_weights, val_weights, test_weights), dim = -1)


torch.Size([2, 1549])
torch.Size([2, 176])
torch.Size([2, 331])


In [ ]:
# Sample negative edges from nodes in each of the sets

from torch_geometric.utils import negative_sampling

def negative_sampling_manual(input_edges,n):

    # This automatically handles the "ensure these are NOT in edge_index" logic
    neg_edges = negative_sampling(
        edge_index=input_edges.t(),       # Your [2, N] positive edges
        num_nodes=n,         # Total unique proteins
        num_neg_samples=5*all_edges.size(1) # Usually 1 negative per 1 positive
    )
    
    mask_col0 = torch.isin(neg_edges[0], train_idx)
    mask_col1 = torch.isin(neg_edges[1], train_idx)
    train_mask = mask_col0 & mask_col1
    
    mask_col0 = torch.isin(neg_edges[0], test_idx)
    mask_col1 = torch.isin(neg_edges[1], test_idx)
    test_mask = mask_col0 | mask_col1
    
    val_mask = ~(test_mask | train_mask)
    
    return neg_edges[:,train_mask], neg_edges[:,val_mask],neg_edges[:,test_mask]
    
neg_train_edges, neg_val_edges,neg_test_edges = negative_sampling_manual(input_edges,n)

print(neg_train_edges.shape)
print(neg_val_edges.shape)
print(neg_test_edges.shape)


torch.Size([2, 6476])
torch.Size([2, 2309])
torch.Size([2, 1495])


# PAF trainer - contains the three models, 
# 1. GAT (goal: get graph-informed embeddings)
# 2. MLP (goal: predicted interaction confidence (or presence/absence interaction ) )
# 3. NF (goal: assign probability to interaction of two nodes)

# Each model is trained separately, with MLP and NF build after GAT
# For training model, the network is augmented from training to test:
# in other words:
# Training is just nodes and edges in training data
# Validation  is practically training + (nodes in ) validation data
# Test data is practically training + validation + (nodes in) test

In [ ]:
# Three sets of LR, L2 decay, and dropout rates for each of the three models
# n_layers for MLP for classifier
# n_flows for RealNVP model
# percentile - threshold to establish an edge as 'positive'

trainer = PAF_trainer(
                 x_esm_embed = input_data,
                 hidden_dim=128,
                 output_dim=64,
                 n_layers = 2,
                 n_flows = 2,
                 percentile = 0.5,
                 lr = [5e-3] * 3,
                 l2_coef = [5e-4] * 3,
                 dropout = [0.2] * 3,
                 device = device)

#trainer.load("GAT","GAT_model.pth")
#trainer.load("MLP","MLP_model.pth")
#trainer.load("RealNVP","NVP_model.pth")

Loaded checkpoint ← GAT_model.pth
Loaded checkpoint ← MLP_model.pth
Loaded checkpoint ← NVP_model.pth


# Graph attention network model training
# Includes CL to minimize distance between graph network embeddings and ESM embeddings (input: ESM embeddings + edges + weights, output: graph attention network embeddings)

In [227]:
# Step 1 - use contrastive learning to learn attention-based graph network from embeddings
best_cl = trainer.train_embeddings(input_data, train_edges, train_weights,train_val_edges,train_val_edge_weights,
                                   epochs= 50)

Training Progress:   0%|          | 0/50 [00:00<?, ?epoch/s]/Users/jam/Documents/PAF/PAF_model.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.device):
/Users/jam/Library/Python/3.9/lib/python/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Training Progress: 100%|██████████| 50/50 [00:02<00:00, 20.50epoch/s, Epoch=50/50, Tr=2.5536, Vall=2.5533]


In [237]:
#trainer.save("GAT", "GAT_model.pth")
trainer.load("GAT","GAT_model.pth")

Loaded checkpoint ← GAT_model.pth


In [194]:
test_cl = trainer.valid_step_embeddings(input_data, all_edges,all_edge_weights)
train_cl = trainer.valid_step_embeddings(input_data, train_edges, train_weights)
val_cl = trainer.valid_step_embeddings(input_data, train_val_edges, train_val_edge_weights)

print(train_cl)
print(val_cl)
print(test_cl)


tensor(2.5446)
tensor(2.5522)
tensor(2.5566)


In [ ]:
mask_col0 = torch.isin(train_edges[0], torch.tensor([1]))
mask_col1 = torch.isin(train_edges[1], torch.tensor([1]))

query_mask = mask_col0 | mask_col1
query_edges = train_edges[:,query_mask]
query_weights = train_weights[query_mask]

## ESM embeddings
input_data[1]
## Graph embedding
trainer._get_embeddings(input_data, edge_index = query_edges, edge_weights = query_weights)[1]
## ESM -> Graph Embeeding
trainer._get_embeddings(input_data[1])

tensor([ 0.0145,  0.0982,  0.2175, -0.0643, -0.0825,  0.0112,  0.0112, -0.1293,
         0.1893, -0.1112,  0.0101,  0.1111,  0.0435,  0.0399, -0.1170,  0.0346,
         0.1842, -0.3144, -0.2083, -0.0052, -0.0617, -0.0100,  0.1148,  0.0918,
        -0.1947, -0.0252,  0.1984,  0.1303, -0.0348, -0.0403,  0.0346,  0.0060,
        -0.1160,  0.0690, -0.0707, -0.2445, -0.1956,  0.0184, -0.1550,  0.2373,
         0.0369, -0.1198,  0.2628, -0.0978,  0.1309, -0.1335, -0.0527, -0.1207,
         0.0073,  0.0540, -0.1083,  0.0412, -0.0423, -0.1842,  0.2305, -0.0681,
        -0.0459,  0.0301,  0.1361, -0.1091,  0.0993,  0.0378, -0.0267, -0.1770])

# MLP model training
# Confidence level predictor (input: ESM embeddings + edges + edge weights, output: predicted interaction confidence)
# Can be converted to just simple prediction (i.e unweighted edges) (requires changing Softplus -> Sigmoid + l1_loss -> binary loss)

In [229]:
# Train classifier/predictor to learn edge confidence

best_mse = trainer.train_predictor(input_data,
                                   torch.cat((train_edges, neg_train_edges),dim = -1),
                                   torch.cat((train_weights, torch.zeros(neg_train_edges.shape[1])),dim = -1),
                                   torch.cat((val_edges, neg_val_edges),dim = -1),
                                   torch.cat((val_weights, torch.zeros(neg_val_edges.shape[1])),dim = -1),
                                   epochs= 100)

Training Progress:   0%|          | 0/100 [00:00<?, ?epoch/s]/Users/jam/Documents/PAF/PAF_model.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.device):
Training Progress: 100%|██████████| 100/100 [00:08<00:00, 11.94epoch/s, Epoch=100/100, Tr=0.0407, Vall=0.0255]


In [238]:
#trainer.save("MLP", "MLP_model.pth")
trainer.load("MLP","MLP_model.pth")

Loaded checkpoint ← MLP_model.pth


In [197]:
# Get sample embeddings and predict edge confidence 

embs = trainer._get_embeddings(input_data,
                               torch.cat((train_edges, neg_train_edges),dim = -1),
                               torch.cat((train_weights, torch.zeros(neg_train_edges.shape[1])),dim = -1),)
src_indices = torch.cat((train_edges, neg_train_edges),dim = -1)[0] # [E_total]
dst_indices = torch.cat((train_edges, neg_train_edges),dim = -1)[1] # [E_total]

src_embs = embs[src_indices] 
dst_embs = embs[dst_indices]

#embs1 = embs[8].unsqueeze(0)
#embs2 = embs[90].unsqueeze(0)

with torch.no_grad():
    error, preds = trainer.model2(src_embs,dst_embs,torch.cat((train_weights, torch.zeros(neg_train_edges.shape[1])),dim = -1) )

print(preds[1:10].squeeze(-1))
print(train_weights[1:10])

tensor([0.0058, 0.0029, 0.0039, 0.0403, 0.0174, 0.0016, 0.0206, 0.0006, 0.0062])
tensor([0.2991, 0.4293, 0.0345, 0.0175, 0.1444, 0.0080, 0.0848, 0.2278, 0.0112])


In [198]:
# Correlation between predicted values and actual edge weights
vx = preds - torch.mean(preds)
vy = train_weights - torch.mean(train_weights)
correlation = torch.sum(vx * vy) / (torch.sqrt(torch.sum(vx ** 2)) * torch.sqrt(torch.sum(vy ** 2)))

print(f"Correlation: {correlation.item():.4f}")

Correlation: 0.0000


# Normalizing Flows model training
# also probability density estimation (input: embeddings of two nodes, output: probability two nodes have an interaction)

In [264]:
best_nll = trainer.train_flow(input_data,
                                   train_edges,
                                   train_weights,
                                   val_edges,
                                   val_weights,
                                   epochs= 100,
                                   percentile = 0.3 )

Training Progress:   0%|          | 0/100 [00:00<?, ?epoch/s]/Users/jam/Documents/PAF/PAF_model.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.device):
/Users/jam/Library/Python/3.9/lib/python/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Training Progress: 100%|██████████| 100/100 [00:03<00:00, 30.78epoch/s, Epoch=100/100, Tr=138.2559, Vall=162.1942]


In [266]:
trainer.save("RealNVP", "NVP_model.pth")
trainer.load("RealNVP","NVP_model.pth")

Saved checkpoint RealNVP → NVP_model.pth
Loaded checkpoint ← NVP_model.pth


In [ ]:
embs = trainer._get_embeddings(input_data,
                               train_edges,
                               train_weights)

src_indices = train_edges[0] # [E_total]
dst_indices = train_edges[1] # [E_total]

src_embs = embs[src_indices] 
dst_embs = embs[dst_indices]

with torch.no_grad():
    pz = trainer._get_edge_prob(src_embs,dst_embs)

print(pz[1:10])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0.])
